In [0]:
-- ============================================================================
-- NORTH RIVER WAREHOUSE — OLAP star schema
--
-- Why a separate schema from the OLTP source?
--   1. Different workloads: the OLTP schemas are normalized (3NF) for fast
--      transactional writes with minimal duplication; the warehouse is
--      deliberately denormalized (star schema) so BI queries join few,
--      wide tables instead of navigating many normalized ones.
--   2. Isolation: heavy analytical scans never touch the transactional source,
--      so reporting cannot slow down or lock operational writes.
--   3. History: OLTP overwrites current state; the warehouse can keep full
--      history via SCD Type 2 (valid_from / valid_to) on dimensions.
--   4. Rebuildability & governance: the warehouse can be reloaded from staging
--      or evolved (new facts, new attributes) without any change to production
--      tables, and access can be granted per layer.
-- ============================================================================

-- ----------------------------------------------------------------------------
-- Dimensions — descriptive, slowly changing context ("who / what / when")
-- ----------------------------------------------------------------------------
CREATE SCHEMA IF NOT EXISTS northriver_warehouse
  COMMENT 'North River warehouse — OLAP star schema, rebuildable from northriver_oltp staging';

CREATE TABLE IF NOT EXISTS northriver_warehouse.dim_customer (
  customer_sk      BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key (stable join key)
  customer_id      INT,           -- natural key from stg_customers
  first_name       STRING,
  last_name        STRING,
  email            STRING,
  region           STRING,
  segment          STRING,
  effective_from   DATE,         -- SCD Type 2 history tracking
  effective_to     DATE,
  is_current       BOOLEAN
);

CREATE TABLE IF NOT EXISTS northriver_warehouse.dim_product (
  product_sk       BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
  product_id       INT,           -- natural key from stg_products
  product_name     STRING,
  brand            STRING,
  category         STRING,
  subcategory      STRING,
  unit_cost        DECIMAL(10,2),
  effective_from   DATE,         -- captures cost/category changes over time
  effective_to     DATE,
  is_current       BOOLEAN
);

CREATE TABLE IF NOT EXISTS northriver_warehouse.dim_channel (
  channel_sk       BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
  channel_name     STRING NOT NULL  -- natural key from stg_orders.channel
);

CREATE TABLE IF NOT EXISTS northriver_warehouse.dim_date (
  date_sk          BIGINT,       -- deterministic YYYYMMDD key — no identity, dates are immutable
  full_date        DATE,
  year             INT,
  quarter          INT,
  month            INT,
  day_of_week      STRING
);

-- ----------------------------------------------------------------------------
-- Facts — measurable business events (keys + numeric measures only)
-- ----------------------------------------------------------------------------

-- Grain: one row per ORDER LINE ITEM (stg_order_items joined to stg_orders) —
-- the atomic, most detailed fact; roll up to order / product / customer /
-- channel / date from here.
CREATE TABLE IF NOT EXISTS northriver_warehouse.fact_orders (
  order_id         INT,          -- degenerate dimension (no separate dim needed)
  order_item_id    BIGINT,
  customer_sk      BIGINT,
  product_sk       BIGINT,
  channel_sk       BIGINT,
  date_sk          BIGINT,
  quantity         INT,
  unit_price       DECIMAL(10,2),
  revenue          DECIMAL(10,2) GENERATED ALWAYS AS (CAST(quantity * unit_price AS DECIMAL(10,2)))
);

-- Design notes:
--   * Surrogate keys (customer_sk, product_sk, channel_sk) let dimension
--     rows change (SCD2) without breaking fact joins; natural keys are
--     retained for lineage back to the source.
--   * stg_order_items carries quantity/price; stg_orders contributes channel
--     and order_date. Both fold into this single line-item-grain fact.
--   * order_id is carried on the fact as a degenerate dimension —
--     it is high-cardinality and has no attributes, so no dimension table.
--   * dim_date has no SCD columns: dates are immutable, so date_sk is a
--     deterministic YYYYMMDD integer instead of an identity surrogate.